In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_DIR = Path("../assets/data")
PROCESSED_DATA_DIR = Path("../data/processed")

file_path = RAW_DATA_DIR / "photovoltaic_spiring_2022_2023.csv"

df = pd.read_csv(file_path)

df["timestamp"] = (
    pd.to_datetime(
        df["timestamp"],
        unit="D",
        origin="1899-12-30",
    )
    .dt.round("min")
)

df = df.sort_values("timestamp").reset_index(drop=True)

In [16]:
SENTINEL_THRESHOLD = -1e6

sentinel_columns = [
    "Weather_Daily_Rainfall",
    "Hail_Accumulation",
]

for column in sentinel_columns:
    df.loc[df[column] < SENTINEL_THRESHOLD, column] = np.nan

df = df.dropna(subset=["Active_Power"]).reset_index(drop=True)

df = df.drop(columns=["Hail_Accumulation"])

In [17]:
feature_columns = [
    "Wind_Speed",
    "Weather_Temperature_Celsius",
    "Global_Horizontal_Radiation",
    "Wind_Direction",
    "Weather_Daily_Rainfall",
    "Max_Wind_Speed",
    "Air_Pressure",
]

for column in feature_columns:
    df[column] = df[column].fillna(df[column].median())

In [18]:
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["month"] = df["timestamp"].dt.month

time_in_minutes = df["hour"] * 60 + df["minute"]

df["time_sin"] = np.sin(
    2 * np.pi * time_in_minutes / (24 * 60)
)

df["time_cos"] = np.cos(
    2 * np.pi * time_in_minutes / (24 * 60)
)

df = df.drop(columns=["hour", "minute"])

In [19]:
TARGET = "Active_Power"

X_columns = [
    "Wind_Speed",
    "Weather_Temperature_Celsius",
    "Global_Horizontal_Radiation",
    "Wind_Direction",
    "Weather_Daily_Rainfall",
    "Max_Wind_Speed",
    "Air_Pressure",
    "time_sin",
    "time_cos",
    "day_of_year",
    "month",
]

X = df[X_columns]
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

X shape: (22461, 11)
y shape: (22461,)
Missing values in X: 0
Missing values in y: 0


In [20]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

processed_path = (
    PROCESSED_DATA_DIR /
    "photovoltaic_spring_2022_2023_processed.csv"
)

processed_df = pd.concat(
    [df[["timestamp"]], X, y],
    axis=1
)

processed_df.to_csv(processed_path, index=False)

print(f"Processed dataset saved to: {processed_path}")

Processed dataset saved to: ..\data\processed\photovoltaic_spring_2022_2023_processed.csv
